In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Patient-wise multi-class metrics for segmentation (Dice, IoU, Hausdorff).

Folder layout per patient (example):
    <root>/Patient001/
        gts/
            001.png, 002.png, ...
        pre/
            001.png, 002.png, ...
    <root>/Patient002/
        gts/...
        pre/...

What this script does
---------------------
• Loads matching PNGs from each patient's gts/ and pre/ folders
• Remaps grayscale masks -> class IDs using your lookup table
• Computes per-class metrics for each image pair: Dice, IoU, Hausdorff (HD) and HD95
• Aggregates means per patient and overall
• Saves detailed CSVs and prints compact summaries

Notes on Hausdorff
------------------
• HD is computed between object *surfaces* (contours) in pixels.
• If both GT and Prediction are empty for a class: HD = NaN (excluded from mean)
• If exactly one side is empty: HD = +inf (included in mean unless you choose to ignore inf)
• HD95 is the 95th percentile of the symmetric surface distances.

Dependencies
------------
• Required: numpy, pillow (PIL), pandas
• Optional (faster/better morphology): scipy.ndimage
  If SciPy is missing, the code falls back to a slower but safe implementation.

Usage
-----
python seg_metrics_patientwise.py \
    --root /path/to/root \
    --out_dir /path/to/save_csvs \
    --classes 0 1 2 3 4 5 6 7 \
    --hd_ignore_inf   # (optional) ignore infinite HD/HD95 when averaging

"""

from __future__ import annotations
import argparse
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import numpy as np
import pandas as pd
from PIL import Image

# -------------------------------
# Your explicit grayscale -> class-ID mapping
# -------------------------------
GRAY_TO_CLASS: Dict[int, int] = {
    0: 0,    # background
    60: 1,   # class 1
    120: 2,  # class 2
    180: 3,  # class 3
    240: 4,  # class 4
    44: 5,   # class 5
    164: 6,  # class 6
    104: 7,  # class 7
}

# Optional pretty names (unused in math, only for reporting if desired)
CLASS_NAMES: Dict[int, str] = {
    0: "Background",
    1: "Femur",
    2: "Tibia",
    3: "Patella",
    4: "Femoral Cartilage",
    5: "Tibial Cartilage",
    6: "Patellar Cartilage",
    7: "Meniscus",
}

# -------------------------------
# Utilities
# -------------------------------

def remap_with_lookup(mask_img: Image.Image) -> np.ndarray:
    """Convert grayscale mask to class IDs using lookup table."""
    arr = np.asarray(mask_img.convert("L"), dtype=np.uint8)
    lut = np.zeros(256, dtype=np.uint8)
    for gray, cls in GRAY_TO_CLASS.items():
        lut[gray] = cls
    return lut[arr]


def dice_iou_from_binary(gt: np.ndarray, pr: np.ndarray) -> Tuple[float, float]:
    """Compute Dice and IoU for binary masks (bool arrays)."""
    gt = gt.astype(bool)
    pr = pr.astype(bool)
    inter = np.logical_and(gt, pr).sum(dtype=np.int64)
    gt_sum = gt.sum(dtype=np.int64)
    pr_sum = pr.sum(dtype=np.int64)
    union = gt_sum + pr_sum - inter

    # Dice
    if gt_sum + pr_sum == 0:
        dice = np.nan  # both empty -> undefined; will be excluded from means
    else:
        dice = (2.0 * inter) / (gt_sum + pr_sum)

    # IoU
    if union == 0:
        iou = np.nan  # both empty
    else:
        iou = inter / union

    return float(dice), float(iou)


# --- Hausdorff support (with SciPy if available) ---
try:
    from scipy.ndimage import binary_erosion, distance_transform_edt
    _HAVE_SCIPY = True
except Exception:
    binary_erosion = None  # type: ignore
    distance_transform_edt = None  # type: ignore
    _HAVE_SCIPY = False


def _edges_from_mask(mask: np.ndarray) -> np.ndarray:
    """Return edge pixels of a binary mask as a boolean array.
    Uses SciPy if available; otherwise falls back to a simple 4-neighborhood erosion.
    """
    mask = mask.astype(bool)
    if _HAVE_SCIPY:
        eroded = binary_erosion(mask)
        edge = np.logical_and(mask, np.logical_not(eroded))
        return edge
    # Fallback: naive 4-neighborhood erosion via shifts
    up = np.zeros_like(mask); up[1:] = mask[:-1]
    down = np.zeros_like(mask); down[:-1] = mask[1:]
    left = np.zeros_like(mask); left[:,1:] = mask[:,:-1]
    right = np.zeros_like(mask); right[:,:-1] = mask[:,1:]
    eroded4 = mask & up & down & left & right
    edge = mask & (~eroded4)
    return edge


def _surface_distances(a_edge: np.ndarray, b_edge: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Directed distances from edges of A to edges of B, and B to A.
    Returns (dists_A_to_B, dists_B_to_A) as 1D arrays of float.
    """
    if a_edge.sum() == 0 and b_edge.sum() == 0:
        return np.array([]), np.array([])
    if _HAVE_SCIPY and distance_transform_edt is not None:
        # distance to nearest True in b_edge can be obtained via EDT on ~b_edge
        dt_b = distance_transform_edt(~b_edge)
        dt_a = distance_transform_edt(~a_edge)
        d_ab = dt_b[a_edge]
        d_ba = dt_a[b_edge]
        return d_ab.astype(float), d_ba.astype(float)

    # Fallback: vectorized nearest-neighbor distances via brute force on coordinates
    a_coords = np.column_stack(np.nonzero(a_edge))  # (N, 2)
    b_coords = np.column_stack(np.nonzero(b_edge))  # (M, 2)
    if a_coords.size == 0 and b_coords.size == 0:
        return np.array([]), np.array([])
    if a_coords.size == 0:
        return np.array([]), np.zeros(len(b_coords))  # distances from empty set are empty
    if b_coords.size == 0:
        return np.zeros(len(a_coords)), np.array([])

    # Compute squared distances in chunks to avoid huge memory when large
    def nn_dists(src: np.ndarray, dst: np.ndarray, chunk: int = 5000) -> np.ndarray:
        out = np.empty(len(src), dtype=float)
        for i in range(0, len(src), chunk):
            s = src[i:i+chunk]
            # (s,1,2) - (1,d,2) -> (s,d,2)
            diffs = s[:, None, :] - dst[None, :, :]
            d2 = np.sum(diffs * diffs, axis=2)
            out[i:i+chunk] = np.sqrt(np.min(d2, axis=1))
        return out

    d_ab = nn_dists(a_coords, b_coords)
    d_ba = nn_dists(b_coords, a_coords)
    return d_ab, d_ba


def hausdorff_hd_and_hd95(gt: np.ndarray, pr: np.ndarray) -> Tuple[float, float]:
    """Compute symmetric Hausdorff (HD) and 95th percentile HD95 in pixels.
    Inputs are binary masks (bool arrays).
    Returns (HD, HD95) where either may be NaN or +inf per rules above.
    """
    gt = gt.astype(bool)
    pr = pr.astype(bool)

    if gt.sum() == 0 and pr.sum() == 0:
        return float("nan"), float("nan")
    if gt.sum() == 0 or pr.sum() == 0:
        # One empty, one non-empty => infinite distance
        return float("inf"), float("inf")

    a_edge = _edges_from_mask(gt)
    b_edge = _edges_from_mask(pr)

    d_ab, d_ba = _surface_distances(a_edge, b_edge)
    # Concatenate both directions for symmetric HD/HD95
    all_d = np.concatenate([d_ab, d_ba]) if d_ab.size or d_ba.size else np.array([])

    if all_d.size == 0:  # identical empty edges shouldn't happen due to earlier checks
        return float("nan"), float("nan")

    hd = float(np.max(all_d))
    hd95 = float(np.percentile(all_d, 95))
    return hd, hd95


# -------------------------------
# Main computation
# -------------------------------

def collect_pngs(gt_dir: Path, pr_dir: Path) -> List[str]:
    """Return sorted list of common filenames present in both folders."""
    gt_names = {p.name for p in gt_dir.glob("*.png")}
    pr_names = {p.name for p in pr_dir.glob("*.png")}
    common = sorted(gt_names & pr_names)
    return common


def compute_metrics_for_pair(gt_img: Image.Image, pr_img: Image.Image, classes: List[int]) -> Dict[int, Dict[str, float]]:
    """Compute per-class metrics for a single (gt, pred) pair. Returns dict[class_id]->metrics."""
    gt_ids = remap_with_lookup(gt_img)
    pr_ids = remap_with_lookup(pr_img)

    out: Dict[int, Dict[str, float]] = {}
    for c in classes:
        gt_c = (gt_ids == c)
        pr_c = (pr_ids == c)
        dice, iou = dice_iou_from_binary(gt_c, pr_c)
        hd, hd95 = hausdorff_hd_and_hd95(gt_c, pr_c)
        out[c] = {
            "dice": dice,
            "iou": iou,
            "hd": hd,
            "hd95": hd95,
        }
    return out


def safe_mean(values: List[float], ignore_inf: bool = False) -> float:
    """Mean over finite, non-NaN values. If none remain, returns NaN."""
    arr = np.asarray(values, dtype=float)
    if ignore_inf:
        arr = arr[np.isfinite(arr)]
    else:
        arr = arr[~np.isnan(arr)]
    if arr.size == 0:
        return float("nan")
    return float(np.mean(arr))


def run(root: Path, out_dir: Path, classes: List[int], hd_ignore_inf: bool = False) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)

    per_file_rows = []  # one row per (patient, file, class)

    patients = [p for p in sorted(root.iterdir()) if p.is_dir()]
    if not patients:
        raise SystemExit(f"No patient folders found under {root}")

    for patient_dir in patients:
        gt_dir = patient_dir / "gts"
        pr_dir = patient_dir / "pre"
        if not gt_dir.exists() or not pr_dir.exists():
            print(f"[WARN] Skipping {patient_dir.name}: missing gts/ or pre/ folder")
            continue

        common = collect_pngs(gt_dir, pr_dir)
        if not common:
            print(f"[WARN] No common PNGs for {patient_dir.name}")
            continue

        for fname in common:
            gt_img = Image.open(gt_dir / fname)
            pr_img = Image.open(pr_dir / fname)
            metrics = compute_metrics_for_pair(gt_img, pr_img, classes)
            for c, m in metrics.items():
                per_file_rows.append({
                    "patient": patient_dir.name,
                    "file": fname,
                    "class": c,
                    "class_name": CLASS_NAMES.get(c, str(c)),
                    **m,
                })

    # Detailed per-file CSV
    df = pd.DataFrame(per_file_rows)
    if df.empty:
        print("No data to report.")
        return

    csv_detailed = out_dir / "per_file_per_class_metrics.csv"
    df.to_csv(csv_detailed, index=False)
    print(f"Saved: {csv_detailed}")

    # Per-patient, per-class means
    agg_funcs = {
        "dice": lambda x: float(np.nanmean(x)),
        "iou": lambda x: float(np.nanmean(x)),
        # For HD metrics we may optionally ignore +inf before averaging
        "hd": (lambda x: safe_mean(list(x), ignore_inf=hd_ignore_inf)),
        "hd95": (lambda x: safe_mean(list(x), ignore_inf=hd_ignore_inf)),
    }

    patient_class = df.groupby(["patient", "class", "class_name"], dropna=False).agg(agg_funcs).reset_index()
    csv_pc = out_dir / "per_patient_per_class_means.csv"
    patient_class.to_csv(csv_pc, index=False)
    print(f"Saved: {csv_pc}")

    # Per-patient overall means (across classes)
    patient_overall = patient_class.groupby(["patient"], dropna=False)[["dice", "iou", "hd", "hd95"]].mean().reset_index()
    csv_po = out_dir / "per_patient_overall_means.csv"
    patient_overall.to_csv(csv_po, index=False)
    print(f"Saved: {csv_po}")

    # Global overall means
    overall_means = {
        "dice": float(np.nanmean(patient_overall["dice"])) ,
        "iou": float(np.nanmean(patient_overall["iou"])) ,
        "hd": float(np.nanmean(patient_overall["hd"])) ,
        "hd95": float(np.nanmean(patient_overall["hd95"])) ,
    }

    # Print compact summary
    print("\n=== Overall means (averaged over patients) ===")
    for k, v in overall_means.items():
        print(f"{k:>5}: {v:.6f}")

    # Also save a tiny summary file
    pd.DataFrame([overall_means]).to_csv(out_dir / "overall_means.csv", index=False)





In [ ]:
def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Patient-wise multi-class metrics for segmentation PNGs.")
    p.add_argument("--root", type=Path, required=True, help="Root folder containing per-patient subfolders")
    p.add_argument("--out_dir", type=Path, required=True, help="Where to save CSV outputs")
    p.add_argument("--classes", type=int, nargs="*", default=sorted(set(GRAY_TO_CLASS.values())),
                   help="List of class IDs to evaluate (default: all mapped classes)")
    p.add_argument("--hd_ignore_inf", action="store_true",
                   help="If set, ignore +inf HD/HD95 when averaging (per-class/per-patient means)")
    return p.parse_args()


from pathlib import Path

# Define your arguments
pred_root = Path("/gpfs/home/machlm03/Segmentation/OAI_demo/MedSAM_Finetune_OAI_Inference/fold0/9000099_00m_LEFT_SAG_3D_DESS_WE/imgs")  # Replace with actual path
gt_root = Path("/gpfs/home/machlm03/Segmentation/OAI_demo/OAI_TrainTest/V00_00m_test_1.0/9000099_00m_LEFT_SAG_3D_DESS_WE/masks")  # Replace with actual path

out_dir = Path("/path/to/results")  # Replace with actual path
classes = [0, 1, 2, 3, 4, 5, 6, 7]
hd_ignore_inf = True



if __name__ == "__main__":
    args = parse_args()
    run(args.root, args.out_dir, args.classes, hd_ignore_inf=args.hd_ignore_inf)

